<a href="https://colab.research.google.com/github/Noisy77-pixel/urdu-ocr-codesaviours-si26-bilal/blob/main/SI26_Week5_Bilal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install required libraries
!pip install gradio transformers torch pillow sentencepiece

import gradio as gr
from transformers import TrOCRProcessor, VisionEncoderDecoderModel
from PIL import Image
import torch
import os
from google.colab import drive

# 1. Mount Google Drive to access your saved model
drive.mount('/content/drive')

model_path = '/content/drive/MyDrive/SI26-urdu-ocr-model'
print(f"Loading trained model from: {model_path}...")

# 2. Load the fine-tuned model and processor from Drive
processor = TrOCRProcessor.from_pretrained(model_path)
model = VisionEncoderDecoderModel.from_pretrained(model_path)
model.eval()

# 3. Define prediction function
def extract_urdu_text(image):
    """Takes an image, returns extracted Urdu text."""
    if image is None:
        return 'Please upload an image'

    try:
        pixel_values = processor(image, return_tensors='pt').pixel_values
    except Exception:
        pixel_values = processor(image, return_tensors='np').pixel_values
        pixel_values = torch.from_numpy(pixel_values)

    with torch.no_grad():
        generated_ids = model.generate(pixel_values)

    text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
    return text if text else 'Could not extract text from this image'

# 4. Build and launch the Gradio UI
interface = gr.Interface(
    fn=extract_urdu_text,
    inputs=gr.Image(type='pil', label='Upload Urdu Image'),
    outputs=gr.Textbox(label='Extracted Urdu Text', lines=3),
    title='Urdu OCR -- Code Saviours SI-26',
    description='Upload an image containing Urdu text and get the extracted text.',
    examples=[]
)

print("\n🚀 Launching Gradio App!")
interface.launch(share=True)

Mounted at /content/drive
Loading trained model from: /content/drive/MyDrive/SI26-urdu-ocr-model...


Loading weights:   0%|          | 0/480 [00:00<?, ?it/s]


🚀 Launching Gradio App!
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://365cf9b09d3844cc49.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
